# Stage 3 + GARCH probabilistic results overview

Обзор вероятностных прогнозов, построенных как `Student-t(mean = stage3 point forecast, sigma = GARCH volatility, df = GARCH nu)`.

Notebook читает результаты из S3:

`s3://binance-data-downloader/stage3_point_garch_probabilistic_eval/<RUN_ID>/`

Основные артефакты:
- `stage3_point_refit_results.parquet`
- `stage3_garch_probabilistic_results.parquet`
- `stage3_garch_probabilistic_predictions.parquet`
- `run_config.json`

In [ ]:
from __future__ import annotations

import io
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import t


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "build_price_feature_day.py").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with build_price_feature_day.py")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("default")

## Load Results

In [ ]:
BUCKET = "binance-data-downloader"
OUTPUT_SUBDIR = "stage3_point_garch_probabilistic_eval"
RUN_ID = "latest"  # e.g. "20260715_121407"

BASE_PREFIX = f"{OUTPUT_SUBDIR.strip('/')}/{RUN_ID.strip('/')}"
s3 = make_s3_client()


def read_bytes(key: str) -> bytes:
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()


def read_json(key: str) -> dict:
    return json.loads(read_bytes(key).decode("utf-8"))


def read_parquet(key: str) -> pd.DataFrame:
    return pd.read_parquet(io.BytesIO(read_bytes(key)))


point_results = read_parquet(f"{BASE_PREFIX}/stage3_point_refit_results.parquet")
prob_results = read_parquet(f"{BASE_PREFIX}/stage3_garch_probabilistic_results.parquet")
prob_predictions = read_parquet(f"{BASE_PREFIX}/stage3_garch_probabilistic_predictions.parquet")
run_config = read_json(f"{BASE_PREFIX}/run_config.json")

print(f"Loaded from s3://{BUCKET}/{BASE_PREFIX}/")
print("point_results", point_results.shape)
print("prob_results", prob_results.shape)
print("prob_predictions", prob_predictions.shape)
display(pd.DataFrame([run_config]).T.rename(columns={0: "value"}).head(40))

In [ ]:
metric_cols = [
    "horizon", "model_name", "point_model", "garch_model", "distribution", "n_rows",
    "MAE", "RMSE", "NLL", "CRPS_approx",
    "PinballLoss_05", "PinballLoss_25", "PinballLoss_50", "PinballLoss_75", "PinballLoss_95",
    "Coverage90", "IntervalWidth90", "sigma_mean", "sigma_median", "nu",
]
metric_cols = [col for col in metric_cols if col in prob_results.columns]
prob_results = prob_results.copy()
for col in ["horizon", "n_rows"]:
    if col in prob_results.columns:
        prob_results[col] = pd.to_numeric(prob_results[col], errors="coerce")
for col in ["MAE", "RMSE", "NLL", "CRPS_approx", "Coverage90", "IntervalWidth90", "sigma_mean", "sigma_median", "nu"]:
    if col in prob_results.columns:
        prob_results[col] = pd.to_numeric(prob_results[col], errors="coerce")

leaderboard = prob_results.sort_values(["horizon", "NLL", "CRPS_approx", "model_name"], na_position="last")
display(leaderboard.loc[:, metric_cols].head(50))

## Selected Stage 3 Point Models

In [ ]:
point_cols = [
    "horizon", "criterion", "family_label", "selected_model_id", "selected_model_family",
    "selected_n_features", "MAE", "RMSE", "Direction_Accuracy_025", "OOS_R2",
    "pred_mean", "pred_std", "true_mean", "true_std",
]
point_cols = [col for col in point_cols if col in point_results.columns]
display(point_results.sort_values(["horizon", "criterion", "family_label"]).loc[:, point_cols])

## Best GARCH per Stage 3 Model

In [ ]:
best_by_point = (
    prob_results
    .sort_values(["horizon", "point_model", "NLL", "CRPS_approx", "model_name"], na_position="last")
    .groupby(["horizon", "point_model"], as_index=False)
    .first()
)
display(best_by_point.loc[:, metric_cols])

best_by_horizon = (
    prob_results
    .sort_values(["horizon", "NLL", "CRPS_approx", "model_name"], na_position="last")
    .groupby("horizon", as_index=False)
    .first()
)
display(best_by_horizon.loc[:, metric_cols])

## Pivot Tables

In [ ]:
def show_metric_pivot(metric: str, aggfunc: str = "min") -> None:
    if metric not in prob_results.columns:
        print(f"Missing metric: {metric}")
        return
    pivot = prob_results.pivot_table(
        index=["horizon", "point_model"],
        columns="garch_model",
        values=metric,
        aggfunc=aggfunc,
    )
    print(metric)
    display(pivot)


for metric in ["NLL", "CRPS_approx", "Coverage90", "IntervalWidth90"]:
    show_metric_pivot(metric)

## Ranking Charts

In [ ]:
for horizon, frame in prob_results.groupby("horizon"):
    ordered = frame.sort_values("NLL", na_position="last").copy()
    labels = ordered["point_model"].astype(str) + " + " + ordered["garch_model"].astype(str)
    fig, axes = plt.subplots(1, 2, figsize=(18, max(5, 0.35 * len(ordered))))
    axes[0].barh(labels, ordered["NLL"])
    axes[0].invert_yaxis()
    axes[0].set_title(f"Horizon {int(horizon)}: NLL lower is better")
    axes[0].set_xlabel("NLL")
    axes[1].barh(labels, ordered["CRPS_approx"])
    axes[1].invert_yaxis()
    axes[1].set_title(f"Horizon {int(horizon)}: CRPS approx lower is better")
    axes[1].set_xlabel("CRPS approx")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ["NLL", "CRPS_approx", "Coverage90"]):
    if metric not in prob_results.columns:
        ax.axis("off")
        continue
    for horizon, frame in prob_results.groupby("horizon"):
        ax.scatter(frame["sigma_mean"], frame[metric], label=f"h{int(horizon)}", alpha=0.75)
    ax.set_xlabel("sigma_mean")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs mean GARCH sigma")
    ax.legend()
plt.tight_layout()
plt.show()

## Prediction Diagnostics

In [ ]:
available_models = (
    prob_predictions[["horizon", "model_name", "point_model", "garch_model"]]
    .drop_duplicates()
    .sort_values(["horizon", "model_name"])
)
display(available_models.head(100))

SELECT_HORIZON = int(best_by_horizon.iloc[0]["horizon"])
SELECT_MODEL = str(best_by_horizon.iloc[0]["model_name"])
print("Selected for diagnostics:", SELECT_HORIZON, SELECT_MODEL)

In [ ]:
diag = prob_predictions.loc[
    prob_predictions["horizon"].eq(SELECT_HORIZON) & prob_predictions["model_name"].eq(SELECT_MODEL)
].copy()
if "timestamp" in diag.columns:
    diag["timestamp"] = pd.to_datetime(diag["timestamp"], utc=True, errors="coerce")
    diag = diag.sort_values("timestamp")
else:
    diag = diag.sort_values("row_id")

display(diag.head())
display(diag[["y_true", "mean_forecast", "sigma_forecast", "q05", "q50", "q95", "loglikelihood"]].describe())

In [ ]:
plot_frame = diag.copy()
if len(plot_frame) > 2000:
    plot_frame = plot_frame.iloc[-2000:].copy()
x = plot_frame["timestamp"] if "timestamp" in plot_frame.columns else plot_frame["row_id"]

fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(x, plot_frame["y_true"], label="y_true", linewidth=0.8, alpha=0.75)
ax.plot(x, plot_frame["mean_forecast"], label="mean_forecast", linewidth=1.0)
ax.fill_between(x, plot_frame["q05"], plot_frame["q95"], alpha=0.18, label="90% interval")
ax.set_title(f"{SELECT_MODEL}: last {len(plot_frame)} rows")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def add_pit(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    nu = pd.to_numeric(out["nu"], errors="coerce").to_numpy(dtype=float)
    mean = pd.to_numeric(out["mean_forecast"], errors="coerce").to_numpy(dtype=float)
    sigma = np.maximum(pd.to_numeric(out["sigma_forecast"], errors="coerce").to_numpy(dtype=float), 1e-12)
    y_true = pd.to_numeric(out["y_true"], errors="coerce").to_numpy(dtype=float)
    standardizer = np.sqrt(np.maximum(nu - 2.0, 1e-12) / nu)
    z = (y_true - mean) / (sigma * standardizer)
    out["pit"] = t.cdf(z, df=nu)
    return out


diag_pit = add_pit(diag)
coverage90 = ((diag_pit["y_true"] >= diag_pit["q05"]) & (diag_pit["y_true"] <= diag_pit["q95"])).mean()
print(f"Empirical 90% coverage: {coverage90:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(diag_pit["pit"].dropna(), bins=40, range=(0, 1))
axes[0].axhline(len(diag_pit) / 40, color="black", linestyle="--", linewidth=1)
axes[0].set_title("PIT histogram")
axes[0].set_xlabel("PIT")
axes[1].hist(diag_pit["loglikelihood"].dropna(), bins=60)
axes[1].set_title("Log-likelihood distribution")
axes[1].set_xlabel("loglikelihood")
plt.tight_layout()
plt.show()

## Coverage Summary

In [ ]:
coverage_rows = []
for (horizon, model_name), frame in prob_predictions.groupby(["horizon", "model_name"]):
    y = pd.to_numeric(frame["y_true"], errors="coerce")
    row = {
        "horizon": horizon,
        "model_name": model_name,
        "rows": len(frame),
        "coverage_50_q25_q75": float(((y >= frame["q25"]) & (y <= frame["q75"])).mean()),
        "coverage_90_q05_q95": float(((y >= frame["q05"]) & (y <= frame["q95"])).mean()),
        "mean_interval_50": float((frame["q75"] - frame["q25"]).mean()),
        "mean_interval_90": float((frame["q95"] - frame["q05"]).mean()),
    }
    coverage_rows.append(row)

coverage_summary = pd.DataFrame(coverage_rows).sort_values(["horizon", "coverage_90_q05_q95", "mean_interval_90"])
display(coverage_summary)